# RepairScope-X — Notebook 10
## Dream-7B DLM Repair-Surface Pilot

Notebook 09 established that the Dream diffusion runtime and the selective-remasking clamp both work correctly on the available T4 GPU:

- 18/18 DLM repair generations completed;
- minimum actual forward calls = 64;
- mechanically forced token positions were preserved;
- no runtime error or OOM occurred.

The strict `FINAL_ANSWER:` verifier counted **0/18** successes. Inspection, however, showed that this conflated semantic failure with output-format variation. In particular, several SVAMP repairs produced the correct numeric answer `784` but used forms such as `FINAL: 784`, LaTeX, or prose instead of the exact strict marker. One Countdown global generation also produced the correct arithmetic expression and target but not the exact required final-line formatting.

Notebook 10 therefore keeps the original strict verifier **unchanged**, but adds a pre-specified task-aware verifier rather than silently relaxing the old metric.

### Three reported success levels

1. **Strict success** — exact `FINAL_ANSWER: <number>` line, preserving comparability with earlier notebooks.
2. **Numeric success** — final numeric prediction equals the target, ignoring marker formatting.
3. **Task success** — primary DLM pilot outcome:
   - GSM8K / SVAMP: numeric success.
   - Countdown: a valid arithmetic expression must use every supplied number exactly once and evaluate to the requested target.

### Balanced pilot

A deterministic SHA-256 ranking with selection seed `3100` chose **4 non-format-only natural failures from each dataset**:

- 4 Countdown
- 4 GSM8K
- 4 SVAMP

No repair outcome was used for selection.

### DLM repair scopes

All actions use the same 64 reverse denoising steps:

- `open_last_25`: preserve first 75%, denoise the final 25%.
- `open_last_50`: preserve first 50%, denoise the final 50%.
- `open_last_75`: preserve first 25%, denoise the final 75%.
- `global`: denoise the entire fixed-length response canvas.

With 3 new seeds:

**12 failures × 4 actions × 3 seeds = 144 DLM repair generations**

This is still an exploratory **4-bit NF4** DLM condition. It is not the final full-precision paper experiment.

Embedded payload SHA-256: `5e8ef181c162cb5f69e3d14a9211a3ec525d490e066a524b573ac06e52745f84`

In [ ]:
!pip -q install -U "transformers>=4.57,<5" "accelerate>=1.0" "bitsandbytes>=0.47" pandas

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, ast as pyast
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import transformers

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 3100
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Notebook 10 requires a CUDA GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", GPU_NAME)
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if GPU_GB < 14:
    raise RuntimeError(
        "This exploratory 4-bit Dream pilot expects approximately "
        "a T4-class GPU with ~15 GB VRAM or larger."
    )

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError(
        "GPU memory is already occupied. Restart the runtime and run from the top."
    )

print("Runtime preflight: PASS")

## 2. Recover the exact balanced failure set

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "5e8ef181c162cb5f69e3d14a9211a3ec525d490e066a524b573ac06e52745f84"

_raw = zlib.decompress(
    base64.b64decode('''eNrtWwuP2zYS/iuEcUDTi+3woQdl3APbS66XIg16m/SColuolETZgmXR1cPOXrD//WYo2SutH92kGxToHtB6dyXOcEjO981H0vkwqnSu41onYaqyvCl1NZr9+GG0Lk2U61WYJaPZKDZNUSdmW8xilWdRqerMFDMmqTuTo/EoUbWqdN1vGKbgaAHv1qqsM2yOb2+N8U1pVms0+r7SpF5oUjSrSJcVYc6YMDomqkgIF0S/V3GdXxNTxNDOELBLmrg1qVU51zVhU3Kp66YsiCIb6CQhqszqxUrXWQz2a4ilgk6tRw3/b+El+fmfL19fvAovXr959+JyRv7Sdv+3n6cQW+t3NGNTOh6Zpl43GOgTiGYC4X1J/kyeMEqeQXhfXhVDP3TKPSq9wPVcznwRMAH+NrrM0ixuhz6D2VVVpWFmU5VXegwTU+u5Ka+hj21pinkIsYBFDJZVDT/rEIaQwE9rftgDTKYuVxkMcqN/tWlvbDfjEcxto/IwNeVWlUkIS5RDAjhsPMoKGHVYm6Uu4AmjfDya60LD+kGu7B5jwy3YhJWOTZHAEz71XDcQHncYc4KAchudWoabUq3CeTSauVPfk4JK5nFPBpwG49HKJDpvc+3fW108ww8+dSfiq8nLAqagievRrlWpN1nVJpRSUvvcFX4QiEgFgfaCJFUqVW6gE8rciPkudRIGtmVThFU2LxSkiQZT6UQ+48xJHUdrLaIgCWjgxqkTp27kJzyKfSXjlEeaJdzllFKPcj8JGI2UR2PHLg04gxGPzBL+0mVpytGsaPL8Znx//HD3MwAoGBMedABy7gUg57MhyLkDoQAhFCCEILRn5Al8TiDaAxxNmPPQwJm0wZzASve2H/hJfMDsHuBDHMMHNhzig00dh0rhsYAx3xee5Efx4XDKWeBwJqT/mPHh0IfHB2Kjg4e8Fzr4Z0MHP4IOCw4J2GD0Lig4fWhMcHoGEu3LfrCnESEOEcGOIkIcIkJICdXJlZR5gef7/lFEsMCnbuAJJhz5mBHxGQoGh3rhdPXiXohwP1u9cO/WC4nlwUoufhcND14gzpWHQXFwzxUHFtxTPLGD4kCnAfNBpbFAOj6jXvB/8TSaVyu5nNWlyoqZLxgbpL99eb+kvyBVA2u70KUhsSpIAyBonzQr+LNaa51gusNggXnJKst1RbKCOPBr0dS6mpK3i85kk+U5xEMWqoIUJy/g78m3JgITQImqSVJC+oAQozs/YEMWpimn5F9mS6C/6+5FCoFDSDaieufexohxZIgga4h+ey12AaCZ7ezvfQzRAYbe3nULgbl347oqDgZ3ehBXxfMsTXWpgSFm6GxiW/2VTGTn+Ko40HPSfXA9J91zeq5925+U0/sd/wCyzDu63/EP9zuODJjDfMo9F2qU7x6DbCDhvWQo6VDwPSLIesyVnwrZb0yl14sWZaTUaZnNcT1MCbWKbGFhIB+1/YFYzCrMTkgjW3UAGPZgA7KGmI0uWmDColWQUpDLZgtmCNUuk2uCK92BvDa1yolaYXElJiUrU+jr1kMX01pdA3hNaSGJaTK/RkZJSHRtH/WjJRBZvSg1+M2wb3zf+YbfBp7asPqO+sOckpdp2+FgZNB8N6CtATohf3IBjcgdGHi9GJN6m0EB356Kv7W925v1OwaKyeMmhwetAOjPzK9GDxQ7VwngrxqoXghvwFCvdP1FRRJdmK6Xzv/pWE/NEIFUubp6Qt7B55dTchHHpkyyYo7Ebq3bNB1/3BrcXUzsQdgu2ly755KMYQWyeGHT4XZBPjYFID8hHlNCnm+1LQBbkF56hpR7dfUj4e+AiDEBrq5+wmdvTL7BGUD/+6nZt8bGV1dpqeIPYHPzgd/AA+72rMe/cUWy6qN431I5MP9KlUtdhjC94T1KAdLQ6W2MO9zH2PQ7Ld+4PCgG7tFiYFsOq4E3xR27hzt3HjDuHakFviMdj0JJ8FwhHlctAEnrfLp8w6lY2oSPVVleI7up99mqWSFDMzHGnF/OK/wLdMw6z5DNf9AVJGGirts0bn1sgSVyoxLdbUxQvETmvaVmQORWZ/P2IbrTKl5Y1BVjktVHbElsK0cH7tbYG9ga2//QWM1RY1kXgLZKxcuhB3i4zHIzh8zZ+dlLx0iBb9sMge3cbdmSAqi5GpSc7Q+LwX74fanIhkS8n60v2jhnyBjMcklbulqZB5+UAovgGEEM/sOO/4u+xd7Aw/Zev7Wdjb5/97a1S1v26TVHAyw5drg1qQxK5l1cEM1T9A6faIWhCTa0vtQrmGicp1itVZzV162xwPaTtj3YOUOr13Zr+nsTl0OHxPXRtOXfm7bcAMiIU+Aul7pMHOWtgAIZOL7PA+dxbTthHy5+w7YzXhiTQ/qlGsCVqVYczEsoqAnuLdtTltiYJclVpWC8lVUVsalrNYf6mnWyFPaMaqktBncNd5ICN7McKjG4tAQ48I6cMD5w2TcV50z3qvN2BKjIbcFHEJ0wRcpB5I5BlXe81e8ehLcVTu2ghpz0K+qwPTfCDgcOW+n33gq/d5osC+jW8vkw9hXQ7505tKTZIHpPTuIYmedWAA9GamcC1dB++VqN2CdOjsTWckzbQbvRKPfsdNzjYICt15a3uGUtMfT4Zr9mfcM238TJgZ2Z0m4+bUGJ9O1WINlN9q16FKgexe/NmGLIlx9LmELekzD9qeP4jiOBsEDw0eOEKXxJJee+KwLq/eEJs9qo1bojzHih8kngDsWebXA/0uQB5NpqjWfTW7s1N1vECFKY5w1fLbIlvuoEzsqUiKYp4d5RD10zlQIXFMYU/fM42FKds+lctwdxd/30+UsM2Gt4Cvbw3wZg4swZGBvgQZxBwyEW6FEsyMMjayEF9SA9BZNc+IIePQCTge97DqegM/hjxIITfCoWXjVLPEXJCmBd5nhA41lR25NpS/QtlQNxK2i00rZY46Y8V9do4Ppti6ptAvmzydqWt5nfecSCjGfAMQKhS3Bs2M9t2NVyfia/4b378HeU6PTcNeXu/Z0oT1P/wddbguNKmR1mu+swVzIBHUjJESOHue5DEvqg3V3GvcB9hMnuiU9N9gskVEjs2tI9HiGVNVCzBslltaLlZVDLlW6l4Fewn4WmC7Uia5WrWHfnY2NyCeDQsPsFD4gcKwQdl+Ds1gYYPkYIdKeCB16IKbrjQlgO0wA21LVFloTKtNZmnevWk05OOMCNfYchyP2aSBd9VLeg2wdy3o/pDq7BuI9DXw7vSd/CtjhDHmjqW7l90EXP/XdD99VaxwhYYk8qtiAIu9uwqolgZeP6jkrcu05h5Q6man8o2orlI2Zm0x047+Zl7+feJtP+8eROmeKdc1bN2rNN0N76ff3hP3sP3QVab6w3VsDaZu0xw747WLyuoxuQ2ydc9Qfda3bHlR2bsdd8rb/ubPVrILICYp2Qk31DeJBy2ORc13h+8bufVfQJ2KbnRylvzu59VBEErpAMhIRPHc6dY8qbw38CfHqOoP7jI2CHfbLy/sZUeOGW4LW3KqPuDhzv1mKT4xVHZkWzJrkBVmPevhVCPrVaRAJ5bCFN8dRiqK53bTulAS707l4bf4VH1kOf6OQZsTGRD3+jfPZCuZ/i8kyCH9wmB0e1tH+oLhz8UqDnS1DfnLni6Nc/fOkx6QhG8Ysif7jk/mk8woOeyJglDcI5TBiuKiyVstQBE9kU2S+NDu0jGHf3tGriWFdV2uAo1yorcT5hgeoQJEBVh5CPYddmnxyrrMD7g7BbxSKFvjwglJUGGBWQVDkmQKXLjU2vEIIFmDUaQwxhVvBoJ7TBV7sXvzQKHv+3S8dRVETg1gmdKKtD/X6dG3uJeA2Dfv7q2/Dy+9dvX377Irx8cfH8h50LfPHi4vLVD/D4u4uXl+Gbl19D2ndR3wzmB7CVpbqqcY7iUtv0auoYt9CUexNGJ4y/Zf5MMPzOksM47NSeUjqj9Mi6sihxWcyZTISvA6mCVDmeSESUerESKohlEnki8BKuhFCeq1zqxlGkY80jljLZ8khtgClCgGWXaAmEtZok+WqCh13VclKtIP8nG8wsDVU+SSDmtbrG+4CwWijuemjlOEwHHsTBIbEclcRcQWLFNIqVZDLmQieSe76nfcf1IyqiKEqhMDAtoIH0RwNgPLcxmHL+rP1tQ/fYmPhfHYMHBfA5cRQgJHxPeAFP4jRIpC/8GPaxEQyZK0/p0ccsubJMgv/AAmgky+13v/EXu2Wa5yZS+QjyH8QtZvWPPKBsDB8cnuHxZ2ZrdlXrddXmaa3xtKJbPjrl8MSsw7X9HhnsO1Q+h3hAMZdmbbvP5yGa4HuAxnzdoHLUVa7IWwQlPAit/LJUw5yp6wnm432mzyR+rwWGES8wt6aMTenTuGECI4fKU1RAgUCsCGmw86fe6Gbc/dMSRA4OaTTDu5ab/wHSfJWK''')
)

assert hashlib.sha256(_raw).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(_raw.decode("utf-8"))
FAILURES = UPSTREAM["selected_failures"]
UPSTREAM_DLM_GATE = UPSTREAM["notebook09_gate"]

assert len(FAILURES) == 12
assert len({x["problem_id"] for x in FAILURES}) == 12
assert Counter(x["dataset"] for x in FAILURES) == {
    "countdown_fresh": 4,
    "gsm8k": 4,
    "svamp": 4,
}
assert all(
    x["verification"]["category"] != "format_only_candidate"
    for x in FAILURES
)
assert UPSTREAM_DLM_GATE["DLM_RUNTIME_READY"] is True

print("Payload SHA-256:", EMBEDDED_PAYLOAD_SHA256)
print("Balanced failures:", len(FAILURES))
print("By dataset:", dict(Counter(x["dataset"] for x in FAILURES)))

for x in FAILURES:
    print(x["problem_id"], "| target =", x["target"])

print("Failure selection validation: PASS")

## 3. Strict and numeric verifiers

The strict verifier is unchanged. Numeric success is reported separately and never overwrites the strict result.

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)

def close_num(a, b):
    return (
        a is not None
        and math.isclose(
            float(a),
            float(b),
            abs_tol=1e-8,
            rel_tol=1e-8,
        )
    )


def strict_final(text):
    vals = FINAL_RE.findall(str(text))

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def last_number(text):
    vals = NUM_RE.findall(
        str(text).replace(",", "")
    )

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def basic_verification(text, target):
    strict = strict_final(text)
    loose = last_number(text)
    target = float(target)

    return {
        "strict_prediction": strict,
        "numeric_prediction": loose,
        "target": target,
        "strict_success": bool(
            close_num(strict, target)
        ),
        "numeric_success": bool(
            close_num(loose, target)
        ),
    }


assert basic_verification(
    "x\nFINAL_ANSWER: 4",
    4,
)["strict_success"]

assert basic_verification(
    "The result is 4.",
    4,
)["numeric_success"]

assert not basic_verification(
    "The result is 5.",
    4,
)["numeric_success"]

print("Strict/numeric verifier tests: PASS")

## 4. Countdown task-aware verifier

For Countdown, numeric target matching alone is insufficient.

The verifier:

1. parses the numbers and target from the benchmark prompt;
2. scans the generated response for arithmetic-only substrings;
3. safely parses candidate expressions with Python AST;
4. allows only `+`, `-`, `*`, `/`, parentheses and unary signs;
5. checks that the multiset of numeric literals equals the supplied numbers exactly once;
6. evaluates the expression and requires it to equal the target.

No arbitrary `eval()` is used.

In [ ]:
COUNTDOWN_PROMPT_RE = re.compile(
    r"Use the numbers\s+(.+?)\s+exactly once\s+"
    r"to produce the target\s+"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+))",
    flags=re.IGNORECASE,
)

ARITH_RUN_RE = re.compile(
    r"[0-9eE+\-*/().=\s]{3,}"
)

ALLOWED_BINOPS = (
    pyast.Add,
    pyast.Sub,
    pyast.Mult,
    pyast.Div,
)

ALLOWED_UNARY = (
    pyast.UAdd,
    pyast.USub,
)


def parse_countdown_prompt(prompt):
    m = COUNTDOWN_PROMPT_RE.search(
        str(prompt)
    )

    if m is None:
        raise ValueError(
            "Countdown prompt format was not recognized."
        )

    source_nums = [
        float(x)
        for x in NUM_RE.findall(
            m.group(1)
        )
    ]

    target = float(
        m.group(2)
    )

    return source_nums, target


def safe_arith_tree_value_and_literals(expr):
    try:
        tree = pyast.parse(
            expr,
            mode="eval",
        )
    except Exception:
        return None

    literals = []

    def walk(node):
        if isinstance(
            node,
            pyast.Expression,
        ):
            return walk(
                node.body
            )

        if isinstance(
            node,
            pyast.Constant,
        ):
            if not isinstance(
                node.value,
                (int, float),
            ):
                raise ValueError

            val = float(
                node.value
            )

            if not math.isfinite(
                val
            ):
                raise ValueError

            literals.append(
                val
            )

            return val

        if isinstance(
            node,
            pyast.BinOp,
        ) and isinstance(
            node.op,
            ALLOWED_BINOPS,
        ):
            a = walk(node.left)
            b = walk(node.right)

            if isinstance(
                node.op,
                pyast.Add,
            ):
                return a + b

            if isinstance(
                node.op,
                pyast.Sub,
            ):
                return a - b

            if isinstance(
                node.op,
                pyast.Mult,
            ):
                return a * b

            if isinstance(
                node.op,
                pyast.Div,
            ):
                if abs(b) < 1e-15:
                    raise ZeroDivisionError
                return a / b

        if isinstance(
            node,
            pyast.UnaryOp,
        ) and isinstance(
            node.op,
            ALLOWED_UNARY,
        ):
            val = walk(
                node.operand
            )

            if isinstance(
                node.op,
                pyast.USub,
            ):
                return -val

            return val

        raise ValueError

    try:
        value = walk(tree)

        if not math.isfinite(
            value
        ):
            return None

        return {
            "value": float(value),
            "literals": literals,
        }

    except Exception:
        return None


def number_multiset_equal(a, b):
    if len(a) != len(b):
        return False

    aa = sorted(
        float(x)
        for x in a
    )
    bb = sorted(
        float(x)
        for x in b
    )

    return all(
        close_num(x, y)
        for x, y in zip(
            aa,
            bb,
        )
    )


def countdown_candidates(text):
    text = (
        str(text)
        .replace("×", "*")
        .replace("÷", "/")
        .replace("\\times", "*")
        .replace("\\div", "/")
    )

    candidates = []

    for m in ARITH_RUN_RE.finditer(
        text
    ):
        run = m.group(0).strip()

        if not any(
            op in run
            for op in [
                "+",
                "-",
                "*",
                "/",
            ]
        ):
            continue

        parts = [
            x.strip()
            for x in run.split("=")
            if x.strip()
        ]

        candidates.extend(
            parts
        )

    # Longest first makes the intended full expression likely to be found first.
    candidates = sorted(
        set(candidates),
        key=len,
        reverse=True,
    )

    return candidates


def countdown_task_verification(
    text,
    prompt,
):
    source_nums, target = (
        parse_countdown_prompt(
            prompt
        )
    )

    checked = []

    for expr in countdown_candidates(
        text
    ):
        parsed = (
            safe_arith_tree_value_and_literals(
                expr
            )
        )

        if parsed is None:
            continue

        uses_numbers_once = (
            number_multiset_equal(
                parsed["literals"],
                source_nums,
            )
        )

        value_correct = (
            close_num(
                parsed["value"],
                target,
            )
        )

        checked.append({
            "expression":
                expr,
            "value":
                parsed["value"],
            "literals":
                parsed["literals"],
            "uses_numbers_once":
                uses_numbers_once,
            "value_correct":
                value_correct,
        })

        if (
            uses_numbers_once
            and value_correct
        ):
            return {
                "task_success":
                    True,
                "valid_expression":
                    expr,
                "expression_value":
                    parsed["value"],
                "source_numbers":
                    source_nums,
                "target":
                    target,
                "checked_candidates":
                    checked,
            }

    return {
        "task_success":
            False,
        "valid_expression":
            None,
        "expression_value":
            None,
        "source_numbers":
            source_nums,
        "target":
            target,
        "checked_candidates":
            checked,
    }


# Regression tests.
_demo_prompt = (
    "Use the numbers 19, 29, and 24 exactly once "
    "to produce the target 14. "
    "Return a valid arithmetic expression."
)

assert countdown_task_verification(
    "The valid arithmetic expression is 29 - 19 + 24 = 14.",
    _demo_prompt,
)["task_success"]

assert not countdown_task_verification(
    "29 - 19 = 10. FINAL_ANSWER: 14",
    _demo_prompt,
)["task_success"]

print("Countdown task verifier tests: PASS")

## 5. Unified task verifier

In [ ]:
def verify_repair(
    text,
    failure,
):
    basic = basic_verification(
        text,
        failure["target"],
    )

    if (
        failure["dataset"]
        == "countdown_fresh"
    ):
        structured = (
            countdown_task_verification(
                text,
                failure["prompt"],
            )
        )

        task_success = bool(
            structured[
                "task_success"
            ]
        )

    else:
        structured = None
        task_success = bool(
            basic[
                "numeric_success"
            ]
        )

    return {
        **basic,
        "task_success":
            task_success,
        "countdown_structured":
            structured,
    }


for failure in FAILURES:
    result = verify_repair(
        failure["output"],
        failure,
    )

    assert not result[
        "task_success"
    ], (
        "A supposedly failed starting response passed "
        f"the task-aware verifier: {failure['problem_id']}"
    )

print("All 12 starting failures remain failures under the task verifier: PASS")

## 6. DLM experimental configuration

In [ ]:
MODEL_ID = "Dream-org/Dream-v0-Instruct-7B"
MODEL_REVISION = "05334cb9faaf763692dcf9d8737c642be2b2a6ae"

ACTIONS = [
    "open_last_25",
    "open_last_50",
    "open_last_75",
    "global",
]

SEEDS = [
    3101,
    3102,
    3103,
]

DENOISING_STEPS = 64
TEMPERATURE = 0.2
TOP_P = 0.95
ALG = "entropy"
ALG_TEMP = 0.0

EXPECTED_TRIALS = (
    len(FAILURES)
    * len(ACTIONS)
    * len(SEEDS)
)

assert EXPECTED_TRIALS == 144

print("Actions:", ACTIONS)
print("Seeds:", SEEDS)
print("Expected DLM repair generations:", EXPECTED_TRIALS)

## 7. Load Dream-7B in exploratory NF4 mode

In [ ]:
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
).eval()

assert hasattr(
    model,
    "diffusion_generate",
)

MODEL_DEVICE = next(
    model.parameters()
).device

print("MODEL LOAD: PASS")
print("Class:", type(model).__name__)
print("Device:", MODEL_DEVICE)
print("Load seconds:", round(time.time() - t0, 2))
print(
    "Allocated GB:",
    round(torch.cuda.memory_allocated()/2**30, 3),
)

## 8. Prompt/canvas helpers

In [ ]:
SYSTEM_PROMPT = (
    "Solve the quantitative problem carefully. "
    "Give a mathematically correct answer. "
    "When possible, end with `FINAL_ANSWER: <number>`."
)


def encode_problem_prompt(problem):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": problem,
        },
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    return {
        "input_ids":
            encoded.input_ids.to(
                MODEL_DEVICE
            ),
        "attention_mask":
            encoded.attention_mask.to(
                MODEL_DEVICE
            ),
    }


def encode_failed_response(text):
    ids = tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids[0]

    if len(ids) < 4:
        raise RuntimeError(
            "Failed response tokenization is unexpectedly short."
        )

    return ids


for f in FAILURES:
    ids = encode_failed_response(
        f["output"]
    )

    print(
        f["problem_id"],
        "| tokens =",
        len(ids),
    )

print("Prompt/canvas preparation: PASS")

## 9. DLM repair-scope masks

In [ ]:
def force_mask_for_action(
    length,
    action,
):
    mask = torch.zeros(
        length,
        dtype=torch.bool,
    )

    if action == "global":
        return mask

    preserve_fraction = {
        "open_last_25": 0.75,
        "open_last_50": 0.50,
        "open_last_75": 0.25,
    }[action]

    cut = max(
        1,
        int(
            math.floor(
                length
                * preserve_fraction
            )
        ),
    )

    mask[:cut] = True

    return mask


for length in [
    8,
    17,
    40,
]:
    for action in ACTIONS:
        mask = force_mask_for_action(
            length,
            action,
        )

        print(
            "length",
            length,
            "|",
            action,
            "| forced",
            int(mask.sum()),
            "| open",
            int((~mask).sum()),
        )

print("Repair-scope mask audit: PASS")

## 10. Actual forward-call counter

In [ ]:
class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(
        self,
        module,
        inputs,
    ):
        self.calls += 1

    def __enter__(self):
        self.calls = 0
        self.handle = (
            self.module
            .register_forward_pre_hook(
                self._hook
            )
        )
        return self

    def __exit__(
        self,
        exc_type,
        exc,
        tb,
    ):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

## 11. DLM selective-remasking runner

In [ ]:
def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def run_dlm_repair(
    failure,
    action,
    seed,
):
    reseed(seed)

    prompt = encode_problem_prompt(
        failure["prompt"]
    )

    original_ids = (
        encode_failed_response(
            failure["output"]
        )
        .to(MODEL_DEVICE)
    )

    response_len = int(
        original_ids.numel()
    )

    force_mask_cpu = (
        force_mask_for_action(
            response_len,
            action,
        )
    )

    force_mask = (
        force_mask_cpu.to(
            MODEL_DEVICE
        )
    )

    prompt_len = int(
        prompt[
            "input_ids"
        ].shape[1]
    )

    def clamp_hook(
        step,
        x,
        logits,
    ):
        generated = x[
            :,
            prompt_len:
            prompt_len + response_len
        ]

        if force_mask.any():
            generated[
                :,
                force_mask
            ] = original_ids[
                force_mask
            ].unsqueeze(0)

        return x

    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()

    with ForwardCounter(model) as fc:
        output = model.diffusion_generate(
            prompt[
                "input_ids"
            ],
            attention_mask=
                prompt[
                    "attention_mask"
                ],
            max_new_tokens=
                response_len,
            output_history=False,
            return_dict_in_generate=True,
            steps=
                DENOISING_STEPS,
            temperature=
                TEMPERATURE,
            top_p=
                TOP_P,
            alg=
                ALG,
            alg_temp=
                ALG_TEMP,
            generation_tokens_hook_func=
                clamp_hook,
        )

    wall = time.time() - t0

    final_ids = output.sequences[
        0,
        prompt_len:
        prompt_len + response_len
    ].detach()

    if force_mask.any():
        assert torch.equal(
            final_ids[
                force_mask
            ],
            original_ids[
                force_mask
            ],
        )

    repaired = tokenizer.decode(
        final_ids.tolist(),
        skip_special_tokens=True,
    ).strip()

    verification = verify_repair(
        repaired,
        failure,
    )

    final_cpu = (
        final_ids.cpu()
    )
    original_cpu = (
        original_ids.cpu()
    )

    position_preservation = float(
        (
            final_cpu
            == original_cpu
        )
        .float()
        .mean()
        .item()
    )

    forced_fraction = float(
        force_mask_cpu
        .float()
        .mean()
        .item()
    )

    return {
        "problem_id":
            failure[
                "problem_id"
            ],
        "dataset":
            failure[
                "dataset"
            ],
        "action":
            action,
        "seed":
            int(seed),
        "response_tokens":
            response_len,
        "forced_tokens":
            int(
                force_mask_cpu.sum()
            ),
        "open_tokens":
            int(
                (~force_mask_cpu).sum()
            ),
        "forced_fraction":
            forced_fraction,
        "strict_success":
            bool(
                verification[
                    "strict_success"
                ]
            ),
        "numeric_success":
            bool(
                verification[
                    "numeric_success"
                ]
            ),
        "task_success":
            bool(
                verification[
                    "task_success"
                ]
            ),
        "verification":
            verification,
        "original_output":
            failure["output"],
        "repaired_output":
            repaired,
        "token_position_preservation":
            position_preservation,
        "actual_forward_calls":
            int(fc.calls),
        "wall_seconds":
            float(wall),
        "peak_vram_gb":
            float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
    }

## 12. One real DLM smoke before the 144-trial sweep

In [ ]:
smoke = run_dlm_repair(
    FAILURES[0],
    ACTIONS[0],
    SEEDS[0],
)

print(
    json.dumps(
        {
            k: v
            for k, v in smoke.items()
            if k not in {
                "original_output",
                "repaired_output",
                "verification",
            }
        },
        indent=2,
    )
)

print("\nREPAIRED OUTPUT\n")
print(smoke["repaired_output"])

assert smoke["actual_forward_calls"] >= DENOISING_STEPS
assert math.isfinite(smoke["wall_seconds"])

print("\nDLM PILOT SMOKE: PASS")

## 13. Output directory and immutable run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook10_Dream_DLM_Surface"
    )
else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook10_Dream_DLM_Surface"
    )

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

signature_payload = {
    "protocol_version":
        "dream-dlm-surface-v1",
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "model_id":
        MODEL_ID,
    "model_revision":
        MODEL_REVISION,
    "quantization":
        "bnb_nf4_4bit_exploratory",
    "actions":
        ACTIONS,
    "seeds":
        SEEDS,
    "denoising_steps":
        DENOISING_STEPS,
    "temperature":
        TEMPERATURE,
    "top_p":
        TOP_P,
    "alg":
        ALG,
    "alg_temp":
        ALG_TEMP,
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

TRIALS_PATH = (
    OUT_DIR
    / "dream_dlm_surface_trials.jsonl"
)

MANIFEST_PATH = (
    OUT_DIR
    / "run_manifest.json"
)

if CLEAR_OLD_RESULTS:
    for p in [
        TRIALS_PATH,
        MANIFEST_PATH,
    ]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    if old.get(
        "run_signature"
    ) != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 10 outputs use a different configuration."
        )

else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "run_signature":
                    RUN_SIGNATURE,
                **signature_payload,
                "gpu":
                    GPU_NAME,
                "gpu_total_gb":
                    GPU_GB,
                "torch":
                    torch.__version__,
                "transformers":
                    transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 14. Resume-safe 144-trial DLM sweep

In [ ]:
def trial_id(
    pid,
    action,
    seed,
):
    return (
        pid
        + "|"
        + action
        + "|"
        + str(seed)
    )


EXPECTED_IDS = {
    trial_id(
        failure[
            "problem_id"
        ],
        action,
        seed,
    )
    for failure in FAILURES
    for action in ACTIONS
    for seed in SEEDS
}

assert len(
    EXPECTED_IDS
) == 144


def load_completed(path):
    done = {}

    if not path.exists():
        return done

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                rec = json.loads(
                    line
                )
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"Malformed JSONL at line {line_number}."
                ) from e

            if rec.get(
                "run_signature"
            ) != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found a trial from another Notebook 10 configuration."
                )

            tid = rec[
                "trial_id"
            ]

            if tid in done:
                raise RuntimeError(
                    f"Duplicate trial_id: {tid}"
                )

            done[
                tid
            ] = rec

    return done


completed = load_completed(
    TRIALS_PATH
)

print(
    "Already completed:",
    len(completed),
    "/ 144",
)

counter = 0

with open(
    TRIALS_PATH,
    "a",
    encoding="utf-8",
) as fp:

    for failure in FAILURES:
        for action in ACTIONS:
            for seed in SEEDS:
                counter += 1

                tid = trial_id(
                    failure[
                        "problem_id"
                    ],
                    action,
                    seed,
                )

                if tid in completed:
                    print(
                        f"[{counter}/144] SKIP {tid}"
                    )
                    continue

                try:
                    rec = run_dlm_repair(
                        failure,
                        action,
                        seed,
                    )

                    rec[
                        "trial_id"
                    ] = tid
                    rec[
                        "run_signature"
                    ] = RUN_SIGNATURE
                    rec[
                        "status"
                    ] = "ok"
                    rec[
                        "error"
                    ] = None

                except torch.cuda.OutOfMemoryError as e:
                    torch.cuda.empty_cache()
                    gc.collect()

                    rec = {
                        "trial_id":
                            tid,
                        "problem_id":
                            failure[
                                "problem_id"
                            ],
                        "dataset":
                            failure[
                                "dataset"
                            ],
                        "action":
                            action,
                        "seed":
                            seed,
                        "status":
                            "oom",
                        "error":
                            str(e),
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                except Exception as e:
                    rec = {
                        "trial_id":
                            tid,
                        "problem_id":
                            failure[
                                "problem_id"
                            ],
                        "dataset":
                            failure[
                                "dataset"
                            ],
                        "action":
                            action,
                        "seed":
                            seed,
                        "status":
                            "error",
                        "error":
                            f"{type(e).__name__}: {e}",
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                fp.write(
                    json.dumps(
                        rec,
                        ensure_ascii=False,
                    )
                )
                fp.write("\n")
                fp.flush()

                if rec[
                    "status"
                ] == "ok":
                    print(
                        f"[{counter}/144] {tid} | "
                        f"strict={rec['strict_success']} | "
                        f"numeric={rec['numeric_success']} | "
                        f"task={rec['task_success']} | "
                        f"NFE={rec['actual_forward_calls']}"
                    )
                else:
                    print(
                        f"[{counter}/144] {tid} | "
                        f"{rec['status']} | {rec['error']}"
                    )

print("DLM surface sweep cell finished.")

## 15. Validate exactly 144 completed DLM trials

In [ ]:
records = load_completed(
    TRIALS_PATH
)

missing = (
    EXPECTED_IDS
    - set(records)
)

extra = (
    set(records)
    - EXPECTED_IDS
)

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError(
        "Sweep incomplete. Rerun the sweep cell; completed trials will be skipped."
    )

if extra:
    raise RuntimeError(
        "Unexpected trial IDs exist."
    )

assert len(records) == 144

rows = [
    records[
        tid
    ]
    for tid in sorted(
        EXPECTED_IDS
    )
]

status_counts = Counter(
    x[
        "status"
    ]
    for x in rows
)

print(
    "Statuses:",
    dict(
        status_counts
    ),
)

if status_counts.get(
    "oom",
    0,
):
    raise RuntimeError(
        "At least one DLM trial OOMed."
    )

if status_counts.get(
    "error",
    0,
):
    raise RuntimeError(
        "At least one DLM trial errored."
    )

TRIAL_DF = pd.DataFrame(
    rows
)

assert len(
    TRIAL_DF
) == 144

assert TRIAL_DF[
    "trial_id"
].nunique() == 144

assert (
    TRIAL_DF[
        "actual_forward_calls"
    ] >= DENOISING_STEPS
).all()

print(
    "144-trial DLM matrix integrity: PASS"
)

## 16. Action-level DLM repair surface

In [ ]:
action_rows = []

for action, group in TRIAL_DF.groupby(
    "action"
):
    action_rows.append({
        "action":
            action,
        "trials":
            len(group),
        "strict_success_rate":
            float(
                group[
                    "strict_success"
                ].mean()
            ),
        "numeric_success_rate":
            float(
                group[
                    "numeric_success"
                ].mean()
            ),
        "task_success_rate":
            float(
                group[
                    "task_success"
                ].mean()
            ),
        "mean_actual_nfe":
            float(
                group[
                    "actual_forward_calls"
                ].mean()
            ),
        "mean_preservation":
            float(
                group[
                    "token_position_preservation"
                ].mean()
            ),
        "mean_wall_seconds":
            float(
                group[
                    "wall_seconds"
                ].mean()
            ),
        "max_peak_vram_gb":
            float(
                group[
                    "peak_vram_gb"
                ].max()
            ),
    })

ACTION_DF = pd.DataFrame(
    action_rows
).sort_values(
    [
        "task_success_rate",
        "mean_actual_nfe",
    ],
    ascending=[
        False,
        True,
    ],
)

display(
    ACTION_DF
)

## 17. Dataset-specific task success

In [ ]:
DATASET_DF = (
    TRIAL_DF.groupby(
        [
            "dataset",
            "action",
        ],
        as_index=False,
    )
    .agg(
        trials=(
            "task_success",
            "size",
        ),
        strict_success_rate=(
            "strict_success",
            "mean",
        ),
        numeric_success_rate=(
            "numeric_success",
            "mean",
        ),
        task_success_rate=(
            "task_success",
            "mean",
        ),
        mean_nfe=(
            "actual_forward_calls",
            "mean",
        ),
        mean_preservation=(
            "token_position_preservation",
            "mean",
        ),
    )
)

display(
    DATASET_DF
)

## 18. Per-failure repair surface and DLM Oracle Repair Headroom

For each failure/action pair, empirical task-success probability is estimated from the three seeds.

The pilot oracle chooses the best action separately for each failure.

`DLM_ORH = mean(per-failure oracle task success) - best fixed action task success`

This is an exploratory headroom diagnostic, not a deployable router.

In [ ]:
surface_rows = []

for (
    pid,
    dataset,
    action,
), group in TRIAL_DF.groupby(
    [
        "problem_id",
        "dataset",
        "action",
    ]
):
    assert len(group) == 3

    surface_rows.append({
        "problem_id":
            pid,
        "dataset":
            dataset,
        "action":
            action,
        "strict_success_rate":
            float(
                group[
                    "strict_success"
                ].mean()
            ),
        "numeric_success_rate":
            float(
                group[
                    "numeric_success"
                ].mean()
            ),
        "task_success_rate":
            float(
                group[
                    "task_success"
                ].mean()
            ),
        "mean_nfe":
            float(
                group[
                    "actual_forward_calls"
                ].mean()
            ),
        "mean_preservation":
            float(
                group[
                    "token_position_preservation"
                ].mean()
            ),
    })

SURFACE_DF = pd.DataFrame(
    surface_rows
)

fixed = (
    SURFACE_DF.groupby(
        "action",
        as_index=False,
    )
    .agg(
        mean_task_success=(
            "task_success_rate",
            "mean",
        ),
        mean_nfe=(
            "mean_nfe",
            "mean",
        ),
        mean_preservation=(
            "mean_preservation",
            "mean",
        ),
    )
    .sort_values(
        [
            "mean_task_success",
            "mean_nfe",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

best_fixed = fixed.iloc[0]

oracle_rows = []

for pid, g in SURFACE_DF.groupby(
    "problem_id"
):
    best = g.sort_values(
        [
            "task_success_rate",
            "mean_nfe",
            "mean_preservation",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    ).iloc[0]

    oracle_rows.append({
        "problem_id":
            pid,
        "dataset":
            best[
                "dataset"
            ],
        "oracle_action":
            best[
                "action"
            ],
        "oracle_task_success":
            float(
                best[
                    "task_success_rate"
                ]
            ),
        "oracle_mean_nfe":
            float(
                best[
                    "mean_nfe"
                ]
            ),
        "oracle_preservation":
            float(
                best[
                    "mean_preservation"
                ]
            ),
    })

ORACLE_DF = pd.DataFrame(
    oracle_rows
)

oracle_mean = float(
    ORACLE_DF[
        "oracle_task_success"
    ].mean()
)

best_fixed_mean = float(
    best_fixed[
        "mean_task_success"
    ]
)

DLM_ORH = (
    oracle_mean
    - best_fixed_mean
)

oracle_action_dist = (
    ORACLE_DF[
        "oracle_action"
    ]
    .value_counts(
        normalize=True
    )
    .to_dict()
)

print(
    "Best fixed action:",
    best_fixed["action"],
)

print(
    "Best fixed task success:",
    round(
        best_fixed_mean,
        4,
    ),
)

print(
    "Oracle task success:",
    round(
        oracle_mean,
        4,
    ),
)

print(
    "DLM Oracle Repair Headroom:",
    round(
        DLM_ORH,
        4,
    ),
)

print(
    "Oracle action distribution:",
    oracle_action_dist,
)

display(
    ORACLE_DF
)

## 19. Same-error recurrence / DLM hysteresis diagnostic

In [ ]:
ORIGINAL_WRONG = {}

for failure in FAILURES:
    original = basic_verification(
        failure[
            "output"
        ],
        failure[
            "target"
        ],
    )

    ORIGINAL_WRONG[
        failure[
            "problem_id"
        ]
    ] = original[
        "numeric_prediction"
    ]


def repaired_numeric(
    verification,
):
    return verification[
        "numeric_prediction"
    ]


def same_wrong_state(row):
    if row[
        "task_success"
    ]:
        return False

    pred = repaired_numeric(
        row[
            "verification"
        ]
    )

    original = ORIGINAL_WRONG[
        row[
            "problem_id"
        ]
    ]

    return (
        pred is not None
        and original is not None
        and close_num(
            pred,
            original,
        )
    )


TRIAL_DF[
    "same_wrong"
] = TRIAL_DF.apply(
    same_wrong_state,
    axis=1,
)

hyst_rows = []

for action, group in TRIAL_DF.groupby(
    "action"
):
    failed = group[
        ~group[
            "task_success"
        ]
    ]

    hyst_rows.append({
        "action":
            action,
        "failed_trials":
            len(failed),
        "same_wrong_rate_all":
            float(
                group[
                    "same_wrong"
                ].mean()
            ),
        "same_wrong_given_failure":
            float(
                failed[
                    "same_wrong"
                ].mean()
            )
            if len(
                failed
            )
            else 0.0,
        "task_success_rate":
            float(
                group[
                    "task_success"
                ].mean()
            ),
        "mean_preservation":
            float(
                group[
                    "token_position_preservation"
                ].mean()
            ),
    })

HYSTERESIS_DF = pd.DataFrame(
    hyst_rows
)

display(
    HYSTERESIS_DF
)

## 20. DLM pilot progression gate

Progress to a larger confirmatory DLM experiment when:

- all 144 trials complete without OOM/error;
- selective token preservation is mechanically valid;
- at least 6 task-successful repairs are observed overall;
- at least 2 datasets contain at least one task-successful repair;
- at least 2 repair scopes have non-zero task success.

The gate does **not** require positive ORH because the immediate question is whether DLM-native selective repair has enough signal to justify confirmation.

In [ ]:
successful_repairs = int(
    TRIAL_DF[
        "task_success"
    ].sum()
)

datasets_with_success = int(
    TRIAL_DF.loc[
        TRIAL_DF[
            "task_success"
        ],
        "dataset",
    ].nunique()
)

actions_with_success = int(
    TRIAL_DF.loc[
        TRIAL_DF[
            "task_success"
        ],
        "action",
    ].nunique()
)

forced_rows = TRIAL_DF[
    TRIAL_DF[
        "forced_tokens"
    ] > 0
]

preservation_ok = bool(
    (
        forced_rows[
            "token_position_preservation"
        ]
        + 1e-12
        >=
        forced_rows[
            "forced_fraction"
        ]
    ).all()
)

DLM_PILOT_GATE = {
    "trials":
        int(
            len(
                TRIAL_DF
            )
        ),
    "unique_trial_ids":
        int(
            TRIAL_DF[
                "trial_id"
            ].nunique()
        ),
    "task_successful_repairs":
        successful_repairs,
    "strict_successful_repairs":
        int(
            TRIAL_DF[
                "strict_success"
            ].sum()
        ),
    "numeric_successful_repairs":
        int(
            TRIAL_DF[
                "numeric_success"
            ].sum()
        ),
    "datasets_with_task_success":
        datasets_with_success,
    "actions_with_task_success":
        actions_with_success,
    "best_fixed_action":
        str(
            best_fixed[
                "action"
            ]
        ),
    "best_fixed_task_success":
        best_fixed_mean,
    "oracle_task_success":
        oracle_mean,
    "dlm_oracle_repair_headroom":
        DLM_ORH,
    "mechanical_preservation_ok":
        preservation_ok,
    "minimum_actual_nfe":
        int(
            TRIAL_DF[
                "actual_forward_calls"
            ].min()
        ),
    "quantization":
        "bnb_nf4_4bit_exploratory",
}

DLM_PILOT_GATE[
    "DLM_SURFACE_PILOT_READY"
] = bool(
    DLM_PILOT_GATE[
        "trials"
    ] == 144
    and DLM_PILOT_GATE[
        "unique_trial_ids"
    ] == 144
    and DLM_PILOT_GATE[
        "mechanical_preservation_ok"
    ]
    and DLM_PILOT_GATE[
        "minimum_actual_nfe"
    ] >= DENOISING_STEPS
    and DLM_PILOT_GATE[
        "task_successful_repairs"
    ] >= 6
    and DLM_PILOT_GATE[
        "datasets_with_task_success"
    ] >= 2
    and DLM_PILOT_GATE[
        "actions_with_task_success"
    ] >= 2
)

print(
    json.dumps(
        DLM_PILOT_GATE,
        indent=2,
    )
)

## 21. Save Notebook 10 artifacts

In [ ]:
action_path = (
    OUT_DIR
    / "dlm_action_summary.csv"
)

dataset_path = (
    OUT_DIR
    / "dlm_dataset_summary.csv"
)

surface_path = (
    OUT_DIR
    / "dlm_repair_surface.csv"
)

oracle_path = (
    OUT_DIR
    / "dlm_oracle_actions.csv"
)

hyst_path = (
    OUT_DIR
    / "dlm_hysteresis.csv"
)

gate_path = (
    OUT_DIR
    / "DLM_SURFACE_PILOT_GATE.json"
)

ACTION_DF.to_csv(
    action_path,
    index=False,
)

DATASET_DF.to_csv(
    dataset_path,
    index=False,
)

SURFACE_DF.to_csv(
    surface_path,
    index=False,
)

ORACLE_DF.to_csv(
    oracle_path,
    index=False,
)

HYSTERESIS_DF.to_csv(
    hyst_path,
    index=False,
)

gate_path.write_text(
    json.dumps(
        DLM_PILOT_GATE,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved Notebook 10 outputs:",
    OUT_DIR,
)

## 22. Artifact manifest

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


artifact_paths = [
    MANIFEST_PATH,
    TRIALS_PATH,
    action_path,
    dataset_path,
    surface_path,
    oracle_path,
    hyst_path,
    gate_path,
]

artifact_manifest = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "run_signature":
        RUN_SIGNATURE,
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "files": {
        p.name:
            sha256_file(
                p
            )
        for p in artifact_paths
        if p.exists()
    },
}

artifact_manifest_path = (
    OUT_DIR
    / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        artifact_manifest,
        indent=2,
    )
)

## 23. ZIP and download all results

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook10_Dream_DLM_Surface_results"
)

zip_path = shutil.make_archive(
    str(
        ZIP_BASE
    ),
    "zip",
    root_dir=
        OUT_DIR,
)

print(
    "Created:",
    zip_path,
)

print(
    "Size MB:",
    round(
        Path(
            zip_path
        ).stat().st_size
        / 1024**2,
        3,
    ),
)

if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(
        zip_path
    )

# What to send back

Send:

`RepairScopeX_Notebook10_Dream_DLM_Surface_results.zip`

The most important outputs are:

- `DLM_SURFACE_PILOT_GATE.json`
- `dlm_action_summary.csv`
- `dlm_dataset_summary.csv`
- `dlm_repair_surface.csv`
- `dlm_oracle_actions.csv`
- `dlm_hysteresis.csv`

If `DLM_SURFACE_PILOT_READY = true`, the next experiment should be an **independent multi-seed DLM confirmation** on the strongest and weakest repair scopes, followed by a full-precision replication on a larger-memory GPU.